In [1]:
from google.colab import files

# This will open a dialog to choose the file from your laptop
uploaded = files.upload()

Saving Finance_Trends.csv to Finance_Trends.csv


In [5]:
import pandas as pd

# قراءة الملف
df = pd.read_csv('/content/Finance_Trends.csv')

# طباعة أسماء الأعمدة الفعلية
print(df.columns.tolist())

['gender', 'age', 'Investment_Avenues', 'Mutual_Funds', 'Equity_Market', 'Debentures', 'Government_Bonds', 'Fixed_Deposits', 'PPF', 'Gold', 'Stock_Marktet', 'Factor', 'Objective', 'Purpose', 'Duration', 'Invest_Monitor', 'Expect', 'Avenue', 'What are your savings objectives?', 'Reason_Equity', 'Reason_Mutual', 'Reason_Bonds', 'Reason_FD', 'Source']


In [9]:
# !pip install -q imbalanced-learn

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import joblib

# 1. Load Data
file_path = '/content/Finance_Trends.csv'
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

# 2. Advanced Risk Profiling Logic (Target)
def assign_risk_profile(row):
    high_risk = row.get('Equity_Market', 0) + row.get('Mutual_Funds', 0)
    low_risk = row.get('Fixed_Deposits', 0) + row.get('Government_Bonds', 0) + row.get('Gold', 0)

    if high_risk > low_risk: return 'Aggressive'
    elif high_risk == low_risk: return 'Moderate'
    else: return 'Conservative'

df['Risk_Class'] = df.apply(assign_risk_profile, axis=1)

# 3. Feature Engineering & Selection
# إضافة متغيرات منطقية قد تحسن التوقع إذا كانت متاحة (نكتفي بالمتاح حالياً مع تنظيفه)
selected_features = ['gender', 'age', 'Objective', 'Duration', 'Expect']
X = df[selected_features].copy()
y = df['Risk_Class']

# 4. Professional Encoding
encoders = {}
X_encoded = X.copy()
categorical_cols = ['gender', 'Objective', 'Duration', 'Expect']

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

# 5. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 6. Apply SMOTE to handle Data Imbalance (The Professional Way)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 7. Model Training with Hyperparameter tuning approach (simplified for quick run)
rf_model_pro = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)
rf_model_pro.fit(X_train_smote, y_train_smote)

# 8. Evaluation
y_pred = rf_model_pro.predict(X_test)
print("--- Professional Model Evaluation (After SMOTE) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

# 9. Save Enterprise Model
joblib.dump(rf_model_pro, 'investment_model_pro.pkl')
joblib.dump(encoders, 'feature_encoders_pro.pkl')
joblib.dump(target_encoder, 'target_encoder_pro.pkl')

print("\n✅ Enterprise Model Saved!")

--- Professional Model Evaluation (After SMOTE) ---
Accuracy: 87.38%

              precision    recall  f1-score   support

  Aggressive       0.15      0.90      0.26        59
Conservative       1.00      0.87      0.93      2341

    accuracy                           0.87      2400
   macro avg       0.57      0.89      0.60      2400
weighted avg       0.98      0.87      0.91      2400


✅ Enterprise Model Saved!


In [10]:
import gradio as gr
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the Enterprise Model and Encoders
model = joblib.load('investment_model_pro.pkl')
encoders = joblib.load('feature_encoders_pro.pkl')
target_encoder = joblib.load('target_encoder_pro.pkl')

gender_choices = encoders['gender'].classes_.tolist()
objective_choices = encoders['Objective'].classes_.tolist()
duration_choices = encoders['Duration'].classes_.tolist()
expect_choices = encoders['Expect'].classes_.tolist()

# 2. Core Logic function for the Dashboard
def generate_portfolio(gender, age, objective, duration, expect):
    try:
        # Encode inputs
        g_enc = encoders['gender'].transform([str(gender)])[0]
        obj_enc = encoders['Objective'].transform([str(objective)])[0]
        dur_enc = encoders['Duration'].transform([str(duration)])[0]
        exp_enc = encoders['Expect'].transform([str(expect)])[0]

        input_df = pd.DataFrame({
            'gender': [g_enc], 'age': [age], 'Objective': [obj_enc],
            'Duration': [dur_enc], 'Expect': [exp_enc]
        })

        # Get Probabilities instead of just hard labels
        probs = model.predict_proba(input_df)[0]
        classes = target_encoder.inverse_transform(model.classes_)

        # Format probabilities for display
        prob_dict = {classes[i]: round(probs[i] * 100, 1) for i in range(len(classes))}
        pred_class = classes[np.argmax(probs)]

        # Business Logic & Asset Allocation
        if pred_class == 'Aggressive':
            allocation = {'EGX Stocks & Equity Funds': 60, 'Gold': 20, 'Fixed Deposits': 20}
            advice = "محفظة هجومية: تناسب طموحك في نمو رأس المال والتخطيط طويل الأجل. التركيز الأكبر على الأسهم (مثل البورصة المصرية EGX) للاستفادة من النمو المركب، مع الاحتفاظ بالذهب للتحوط."
        elif pred_class == 'Moderate':
            allocation = {'Mutual Funds': 40, 'Fixed Deposits': 30, 'Gold': 20, 'Government Bonds': 10}
            advice = "محفظة متوازنة: تجمع بين النمو المعتدل والأمان. توزيع مثالي يقلل المخاطر مع الحفاظ على فرص جيدة لزيادة الثروة بمرور الوقت."
        else:
            allocation = {'Fixed Deposits / Certificates': 50, 'Government Bonds': 30, 'Gold': 20}
            advice = "محفظة دفاعية (متحفظة): الأولوية القصوى لحماية رأس المال. الاعتماد على الشهادات البنكية والسندات الحكومية الآمنة، مع نسبة من الذهب لحفظ القيمة."

        # Generate Professional Pie Chart
        fig, ax = plt.subplots(figsize=(6, 5))
        colors = ['#2b5c8f', '#f4a261', '#e76f51', '#2a9d8f']
        ax.pie(
            allocation.values(),
            labels=allocation.keys(),
            autopct='%1.1f%%',
            startangle=140,
            colors=colors,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
            textprops={'fontsize': 10, 'weight': 'bold'}
        )
        ax.axis('equal')
        plt.title(f"Recommended Allocation ({pred_class})", fontsize=14, weight='bold', pad=20)
        plt.tight_layout()

        # Format ML probabilities output
        prob_text = "\n".join([f"🔹 {k}: {v}%" for k, v in prob_dict.items()])

        return fig, advice, prob_text

    except Exception as e:
        # In case of any error, return empty/error states
        fig, ax = plt.subplots()
        ax.text(0.5, 0.5, "Error rendering chart", ha='center')
        return fig, f"Error: {e}", "Error"

# 3. Design Enterprise GUI using gr.Blocks
with gr.Blocks(theme=gr.themes.Soft()) as dashboard:
    gr.Markdown("<h1 style='text-align: center; color: #2b5c8f;'>🏦 نظام إدارة الثروات والتوصية الاستثمارية</h1>")
    gr.Markdown("<p style='text-align: center;'>أدخل البيانات المالية لتحديد مستوى المخاطرة وتوليد التوزيع الأمثل للمحفظة الاستثمارية بالاعتماد على تعلم الآلة.</p>")

    with gr.Row():
        # Left Column for Inputs
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 📋 بيانات العميل")
            gender_in = gr.Dropdown(choices=gender_choices, label="الجنس (Gender)")
            age_in = gr.Slider(minimum=18, maximum=90, step=1, label="العمر (Age)", value=25)
            obj_in = gr.Dropdown(choices=objective_choices, label="الهدف (Objective)")
            dur_in = gr.Dropdown(choices=duration_choices, label="المدة (Duration)")
            exp_in = gr.Dropdown(choices=expect_choices, label="العائد المتوقع (Expected Return)")
            analyze_btn = gr.Button("تحليل وبناء المحفظة 🚀", variant="primary")

        # Right Column for Outputs
        with gr.Column(scale=2):
            gr.Markdown("### 📊 لوحة التحكم والنتائج (Dashboard)")
            with gr.Row():
                prob_out = gr.Textbox(label="تحليل احتمالات المخاطرة للنموذج (Predict Proba)", lines=4)
            with gr.Row():
                advice_out = gr.Textbox(label="التوجيه المالي (Financial Advice)", lines=3)
            with gr.Row():
                plot_out = gr.Plot(label="التوزيع المئوي للأصول (Asset Allocation)")

    # Connect components
    analyze_btn.click(
        fn=generate_portfolio,
        inputs=[gender_in, age_in, obj_in, dur_in, exp_in],
        outputs=[plot_out, advice_out, prob_out]
    )

# 4. Launch Dashboard
dashboard.launch(share=True)

/tmp/ipykernel_14163/3012021071.py:78: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as dashboard:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5c00ffa12bab3a1e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
